In [1]:
!git clone https://github.com/yohanshin/WHAM.git
%cd WHAM

fatal: destination path 'WHAM' already exists and is not an empty directory.
/content/WHAM


In [2]:
!mkdir -p checkpoints
!mkdir -p dataset/body_models

!pip install gdown
!gdown --id 19qkI-a6xuwob9_RFNSPWf1yWErwVVlks -O checkpoints/wham_vit_bedlam_w_3dpw.pth.tar

!gdown --id 1pbmzRbWGgae6noDIyQOnohzaVnX_csUZ -O dataset/body_models.tar.gz
!tar -xvf dataset/body_models.tar.gz -C dataset/

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=19qkI-a6xuwob9_RFNSPWf1yWErwVVlks
From (redirected): https://drive.google.com/uc?id=19qkI-a6xuwob9_RFNSPWf1yWErwVVlks&confirm=t&uuid=97d79c9f-10f1-423c-a257-cf8ff3dbaf11
To: /content/WHAM/checkpoints/wham_vit_bedlam_w_3dpw.pth.tar
100% 527M/527M [00:14<00:00, 36.0MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1pbmzRbWGgae6noDIyQOnohzaVnX_csUZ
To: /content/WHAM/dataset/body_models.tar.gz
100% 964k/964k [00:00<00:00, 31.7MB/s]
body_models/
body_models/coco_aug_dict.pth
body_models/

In [3]:
!mkdir -p /content/WHAM/dataset/body_models/smpl

from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/SMPL_NEUTRAL.pkl /content/WHAM/dataset/body_models/smpl/

!ls -R /content/WHAM/dataset/body_models/smpl/
%cd WHAM

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Check lại chuồng xem có 'xương' chưa ---
/content/WHAM/dataset/body_models/smpl/:
SMPL_NEUTRAL.pkl
[Errno 2] No such file or directory: 'WHAM'
/content/WHAM


In [4]:
!pip install coremltools chumpy loguru smplx yacs -q

In [11]:
# -------------------------------------------------------------------------
# Fix libs
# -------------------------------------------------------------------------
import inspect
if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec

import numpy as np
np.bool = bool
np.int = int
np.float = float
np.complex = complex
np.object = object
np.unicode = str
np.str = str

import torch
if not hasattr(torch, '_load_is_patched'):
    _original_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_torch_load(*args, **kwargs)
    torch.load = _patched_torch_load
    torch._load_is_patched = True
# -------------------------------------------------------------------------
import torch
import torch.nn as nn
import coremltools as ct
from configs.config import get_cfg_defaults
from lib.models import build_network, build_body_model

# ---------------------------------------------------------
# 1. Define the Wrappers
# ---------------------------------------------------------
from configs import constants as _C

class WHAM_Init(nn.Module):
    def __init__(self, network):
        super().__init__()
        self.network = network

    def forward(self, init_kp, init_smpl):
        b = init_kp.shape[0]

        # 1. Motion Encoder Memory
        h_enc, c_enc = self.network.motion_encoder.neural_init(init_kp.reshape(b, 1, -1))

        # 2. Trajectory Decoder Memory
        d_traj = self.network.trajectory_decoder.regressor.rnn.hidden_size
        h_traj = torch.zeros(3, b, d_traj)
        c_traj = torch.zeros(3, b, d_traj)

        # 3. Motion Decoder Memory
        init_smpl_reshaped = init_smpl.reshape(b, 1, 24, 6)
        h_dec, c_dec = self.network.motion_decoder.neural_init(
            init_smpl_reshaped[:, :, _C.BMODEL.MAIN_JOINTS].reshape(b, 1, -1)
        )
        return h_enc, c_enc, h_traj, c_traj, h_dec, c_dec

class WHAM_Step(nn.Module):
    def __init__(self, network):
        super().__init__()
        self.network = network

    def forward(self, x_step, cam_a_step, prev_kp3d, prev_root, prev_pose,
                h_enc, c_enc, h_traj, c_traj, h_dec, c_dec):

        x_emb = self.network.motion_encoder.embed_layer(x_step.reshape(1, 1, -1))
        (pred_kp3d,), motion_context, (h_enc_out, c_enc_out) = self.network.motion_encoder.regressor(
            x_emb, [prev_kp3d], (h_enc, c_enc)
        )
        motion_context = torch.cat((motion_context, pred_kp3d), dim=-1)

        (pred_vel, pred_root), _, (h_traj_out, c_traj_out) = self.network.trajectory_decoder.regressor(
            motion_context, [prev_root, cam_a_step], (h_traj, c_traj)
        )

        (pred_pose, pred_shape, pred_cam, pred_contact), _, (h_dec_out, c_dec_out) = self.network.motion_decoder.regressor(
            motion_context, [prev_pose], (h_dec, c_dec)
        )

        return (pred_kp3d, pred_root, pred_vel, pred_pose, pred_shape, pred_cam, pred_contact,
                h_enc_out, c_enc_out, h_traj_out, c_traj_out, h_dec_out, c_dec_out)

# ---------------------------------------------------------
# 2. Load the Original WHAM Model
# ---------------------------------------------------------
print("Loading configuration...")
cfg = get_cfg_defaults()
cfg.merge_from_file('configs/yamls/demo.yaml')
cfg.TRAIN.CHECKPOINT = 'checkpoints/wham_vit_bedlam_w_3dpw.pth.tar'

print("Building SMPL and Network...")
smpl = build_body_model('cpu', 1)
network = build_network(cfg, smpl)
network.eval()
network.cpu()

# Force flattening of LSTM parameters to avoid tracer warnings
for module in network.modules():
    if isinstance(module, torch.nn.LSTM):
        module.dropout = 0
        module.flatten_parameters()

print("Wrapping models...")
init_model = WHAM_Init(network).eval()
step_model = WHAM_Step(network).eval()

# ---------------------------------------------------------
# 3. Trace Models with Dummy Data
# ---------------------------------------------------------
print("Tracing Init Model...")
dummy_init_kp = torch.randn(1, 1, 88)
dummy_init_smpl = torch.randn(1, 1, 144)
traced_init = torch.jit.trace(init_model, (dummy_init_kp, dummy_init_smpl))

with torch.no_grad():
    h_enc, c_enc, h_traj, c_traj, h_dec, c_dec = init_model(dummy_init_kp, dummy_init_smpl)

print("Tracing Step Model...")
dummy_step_args = (
    torch.randn(1, 1, 37),   # x_step
    torch.randn(1, 1, 6),    # cam_a_step
    torch.randn(1, 1, 51),   # prev_kp3d
    torch.randn(1, 1, 6),    # prev_root
    torch.randn(1, 1, 144),  # prev_pose
    h_enc, c_enc,            # <--- Passed dynamically! No more hardcoding.
    h_traj, c_traj,          # <--- Passed dynamically!
    h_dec, c_dec             # <--- Passed dynamically!
)
traced_step = torch.jit.trace(step_model, dummy_step_args)

# ---------------------------------------------------------
# 4. Convert to Core ML
# ---------------------------------------------------------
print("Converting Init Model to Core ML (FP16)...")
mlmodel_init = ct.convert(
    traced_init,
    inputs=[ct.TensorType(name="init_kp", shape=dummy_init_kp.shape),
            ct.TensorType(name="init_smpl", shape=dummy_init_smpl.shape)],
    outputs=[
        ct.TensorType(name="h_enc"), ct.TensorType(name="c_enc"),
        ct.TensorType(name="h_traj"), ct.TensorType(name="c_traj"),
        ct.TensorType(name="h_dec"), ct.TensorType(name="c_dec")
    ],
    compute_precision=ct.precision.FLOAT16
)
mlmodel_init.save("WHAM_I.mlpackage")

print("Converting Step Model to Core ML (FP16)...")
mlmodel_step = ct.convert(
    traced_step,
    inputs=[
        ct.TensorType(name="x_step", shape=(1, 1, 37)),
        ct.TensorType(name="cam_a_step", shape=(1, 1, 6)),
        ct.TensorType(name="prev_kp3d", shape=(1, 1, 51)),
        ct.TensorType(name="prev_root", shape=(1, 1, 6)),
        ct.TensorType(name="prev_pose", shape=(1, 1, 144)),
        ct.TensorType(name="h_enc_in", shape=h_enc.shape),
        ct.TensorType(name="c_enc_in", shape=c_enc.shape),
        ct.TensorType(name="h_traj_in", shape=h_traj.shape),
        ct.TensorType(name="c_traj_in", shape=c_traj.shape),
        ct.TensorType(name="h_dec_in", shape=h_dec.shape),
        ct.TensorType(name="c_dec_in", shape=c_dec.shape)
    ],
    outputs=[
        ct.TensorType(name="pred_kp3d"), ct.TensorType(name="pred_root"),
        ct.TensorType(name="pred_vel"), ct.TensorType(name="pred_pose"),
        ct.TensorType(name="pred_shape"), ct.TensorType(name="pred_cam"),
        ct.TensorType(name="pred_contact"),
        ct.TensorType(name="h_enc_out"), ct.TensorType(name="c_enc_out"),
        ct.TensorType(name="h_traj_out"), ct.TensorType(name="c_traj_out"),
        ct.TensorType(name="h_dec_out"), ct.TensorType(name="c_dec_out")
    ],
    compute_precision=ct.precision.FLOAT16
)
mlmodel_step.save("WHAM_S.mlpackage")

2026-03-27 10:48:20.794 | INFO     | lib.models:build_network:36 - => loaded checkpoint 'checkpoints/wham_vit_bedlam_w_3dpw.pth.tar' 
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 233.07 passes/s]


In [12]:

!zip -r WHAM_I.zip WHAM_I.mlpackage
!zip -r WHAM_S.zip WHAM_S.mlpackage
